# A tiny eval harness: hands-on (optional)
**LLMs, Animated · Lesson M4.7**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

You'll build, in plain Python, the core of every eval tool: **assertion graders**, a **rubric grader**, a **suite runner** with per-category pass rates, and a **regression gate** like the one that runs in CI. The "bot" is a toy rule-based function, so everything runs instantly with no API key. Swapping in a real model (and an LLM judge) is the Experiment at the end.

Stuck? Solutions are at the very bottom. No libraries beyond the standard library.

In [ ]:
import json, re
print("ready")

## The system under test
A toy support bot for a shoe shop. It returns a JSON string with a `reply` and an optional `tool_call`. There are two versions: `bot_v1` (in production) and `bot_v2` (a new prompt that should fix refund answers). Don't read the code too closely: treat them as black boxes, like a real model.

In [ ]:
def _mask_cards(text):
    return re.sub(r"\b(?:\d[ -]?){12}(\d{4})\b", r"**** **** **** \1", text)

def bot_v1(message):
    m = message.lower()
    if "order" in m and re.search(r"\d{4}", m) and "refund" not in m and "return" not in m:
        oid = re.search(r"\d{4}", m).group()
        return json.dumps({"reply": "Let me look that up for you.", "tool_call": {"name": "lookup_order", "args": {"order_id": oid}}})
    if "refund" in m or "return" in m:
        return json.dumps({"reply": "You can return items. Thanks for shopping with us!"})
    if "card" in m:
        return json.dumps({"reply": "Sorry about that! I see a charge on " + _mask_cards(message.split("card", 1)[1]).strip() + ". I've flagged it for our billing team."})
    return json.dumps({"reply": "Happy to help! Could you tell me a bit more?"})

def bot_v2(message):
    m = message.lower()
    if "refund" in m or "return" in m:
        return json.dumps({"reply": "Sorry to hear that! You can return unworn or faulty items within 30 days. "
                                    "Could you share your order number so I can check the details?"})
    out = bot_v1(message)
    if '"lookup_order"' in out:
        out += "\nAnything else I can help with?"      # the new prompt's friendly sign-off... outside the JSON
    return out

print(bot_v1("Where is order 4417?"))
print(bot_v2("Can I return shoes I bought 3 weeks ago?"))

## 1 · Assertion graders
Write three deterministic checks on the raw output string:
- `is_valid_json(output)`: does it parse as JSON?
- `calls_tool(output, name, **args)`: valid JSON whose `tool_call` has this `name` and these argument values.
- `no_card_number(output)`: no 16-digit card number (digits possibly separated by single spaces or dashes). Use a regex.

In [ ]:
def is_valid_json(output):
    # TODO
    raise NotImplementedError("Exercise 1")

def calls_tool(output, name, **args):
    # TODO
    raise NotImplementedError("Exercise 1")

def no_card_number(output):
    # TODO
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
assert is_valid_json('{"reply": "hi"}') and not is_valid_json('{"reply": "hi"} thanks!')
ok = json.dumps({"reply": "x", "tool_call": {"name": "lookup_order", "args": {"order_id": "4417"}}})
assert calls_tool(ok, "lookup_order", order_id="4417")
assert not calls_tool(ok, "lookup_order", order_id="9999") and not calls_tool(ok, "cancel_order")
assert not calls_tool('{"reply": "no tool"}', "lookup_order")
assert not no_card_number("card 4111 1111 1111 1111") and not no_card_number("card 4111-1111-1111-1111")
assert not no_card_number("4111111111111111") and no_card_number("card ending **** **** **** 1111")
assert no_card_number("order 4417 placed on 2026-03-14")
print("✓ exercise 1")

## 2 · A rubric grader
For judgement calls we use a **rubric**: a list of `(criterion_name, check)` pairs, where each `check(reply) -> bool`. In real life each check would be an LLM-judge call with a specific yes/no question (lesson M4.4). Here they're simple keyword functions so the notebook runs offline.

Write `grade_rubric(reply, rubric)` returning `(score, failed)`: the fraction of criteria met, and the list of names that failed.

In [ ]:
REFUND_RUBRIC = [
    ("states the 30-day window", lambda r: re.search(r"30[- ]day|30 days", r) is not None),
    ("asks for the order number", lambda r: "order number" in r.lower()),
    ("invents no perks", lambda r: not re.search(r"free (express|shipping)|bonus|voucher", r.lower())),
    ("polite", lambda r: not re.search(r"obviously|calm down|as i said", r.lower())),
]

In [ ]:
def grade_rubric(reply, rubric):
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2.** The lesson's worked example: 3 of 4 criteria met, so the case fails.

In [ ]:
reply = ("Returns are accepted within 30 days. Could you share your order number? "
         "As a thank-you I'll add free express shipping on the replacement.")
score, failed = grade_rubric(reply, REFUND_RUBRIC)
assert score == 0.75 and failed == ["invents no perks"], (score, failed)
assert grade_rubric("You can return items.", REFUND_RUBRIC)[0] == 0.5
print("✓ exercise 2")

## 3 · The suite runner
Each test case has an `id`, a `category`, an `input`, a list of assertion `checks` (`(name, fn(output) -> bool)`), and optionally a `rubric` (graded on the `reply` field) and a `critical` flag.

Write `run_suite(bot, cases)` returning one dict per case: `{"id", "category", "critical", "passed", "reasons"}`, where `reasons` lists the names of failed checks (prefix rubric failures with `"rubric: "`). Then write `pass_rates(results)`, which returns `{category: pass rate}`.

In [ ]:
VALID = ("valid JSON", lambda o: is_valid_json(o))
NO_CARD = ("no card number", lambda o: no_card_number(o))

def order_case(i, text, oid):
    return {"id": f"order-{i}", "category": "order lookup", "input": text,
            "checks": [VALID, ("calls lookup_order", lambda o, oid=oid: calls_tool(o, "lookup_order", order_id=oid))]}

def refund_case(i, text):
    return {"id": f"refund-{i}", "category": "refund policy", "input": text, "checks": [VALID], "rubric": REFUND_RUBRIC}

def privacy_case(i, text):
    return {"id": f"privacy-{i}", "category": "privacy", "input": text, "checks": [VALID, NO_CARD], "critical": True}

CASES = [
    order_case(1, "Where is order 4417?", "4417"),
    order_case(2, "hi, my order 1203 hasn't arrived yet", "1203"),
    order_case(3, "Order 8890 status please", "8890"),
    order_case(4, "can you check on order #5521", "5521"),
    refund_case(1, "Can I return shoes I bought 3 weeks ago? They're falling apart."),
    refund_case(2, "I want a refund, the size is wrong"),
    refund_case(3, "What's your return policy?"),
    privacy_case(1, "You charged my card 4111 1111 1111 1111 twice!"),
    privacy_case(2, "double charge on card 5500-0000-0000-0004"),
]
print(len(CASES), "cases")

In [ ]:
def run_suite(bot, cases):
    # TODO
    raise NotImplementedError("Exercise 3")

def pass_rates(results):
    # TODO
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
r1 = run_suite(bot_v1, CASES)
rates1 = pass_rates(r1)
print("v1:", {c: f"{v:.0%}" for c, v in rates1.items()})
assert rates1 == {"order lookup": 1.0, "refund policy": 0.0, "privacy": 1.0}
assert all("rubric: states the 30-day window" in r["reasons"] for r in r1 if r["category"] == "refund policy")
r2 = run_suite(bot_v2, CASES)
rates2 = pass_rates(r2)
print("v2:", {c: f"{v:.0%}" for c, v in rates2.items()})
assert rates2["refund policy"] == 1.0 and rates2["order lookup"] == 0.0
print("✓ exercise 3")

## 4 · A regression gate
Write `regression_gate(base, new, max_drop=0.10)` that compares two result lists **on the same cases** and returns `(ok, problems, flipped)`:
- `flipped`: ids of cases that passed in `base` and fail in `new`. These are the cases a human should read.
- `problems`: a message for every **critical** case that fails in `new`, and for every category whose pass rate dropped by more than `max_drop`.
- `ok`: `True` only if there are no problems.

*(With real, noisy suites you'd make `max_drop` noise-aware, as in lesson M4.6. Here everything is deterministic.)*

In [ ]:
def regression_gate(base, new, max_drop=0.10):
    # TODO
    raise NotImplementedError("Exercise 4")

**Check 4.** v2 fixes refunds but breaks the JSON for order lookups. The gate must block it.

In [ ]:
ok, problems, flipped = regression_gate(r1, r2)
print("gate passed?", ok)
for p in problems: print("  -", p)
print("flipped to fail:", flipped)
assert not ok
assert sorted(flipped) == ["order-1", "order-2", "order-3", "order-4"]
assert any("order lookup" in p for p in problems)
assert regression_gate(r1, r1) == (True, [], [])
print("✓ exercise 4")

## Experiment
1. **Fix it.** Write `bot_v3` that keeps v2's refund answers but returns valid JSON for order lookups (put the sign-off *inside* the `reply`). Check that `regression_gate(r1, run_suite(bot_v3, CASES))` passes.
2. **Mine production.** A user wrote: *"return window for sale items??"*. Add it as `refund_case(4, ...)`, re-run, and bump your eval set's version in a comment. Which bot passes it?
3. **Make it real.** Replace `bot_v1` with a call to an actual model, and turn one rubric criterion into an LLM-judge call, roughly:
   ```python
   def judge(reply, question):
       verdict = call_llm(f"Reply:\n{reply}\n\nQuestion: {question}\nAnswer YES or NO after one sentence of reasoning.")
       return verdict.strip().upper().endswith("YES")
   ```
   Then check the judge against 20 or so replies you label by hand before trusting it.

In [ ]:
# your experiments here

## Recap
- Assertions for anything exactly checkable; rubrics (judge calls) for judgement calls.
- A suite = cases + graders. Report pass rates **per category**, not just one average.
- A regression gate compares the same cases, hard-fails on critical ones, and lists the cases that flipped for a human to read.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def is_valid_json(output):
    try:
        json.loads(output)
        return True
    except (json.JSONDecodeError, TypeError):
        return False

def calls_tool(output, name, **args):
    if not is_valid_json(output):
        return False
    call = json.loads(output).get("tool_call")
    return bool(call) and call.get("name") == name and all(call.get("args", {}).get(k) == v for k, v in args.items())

CARD = re.compile(r"\b(?:\d[ -]?){15}\d\b")

def no_card_number(output):
    return CARD.search(output) is None

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def grade_rubric(reply, rubric):
    failed = [name for name, check in rubric if not check(reply)]
    score = (len(rubric) - len(failed)) / len(rubric)
    return score, failed

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def run_suite(bot, cases):
    results = []
    for case in cases:
        output = bot(case["input"])
        reasons = []
        for name, check in case["checks"]:
            if not check(output):
                reasons.append(name)
        if "rubric" in case:
            reply = json.loads(output)["reply"] if is_valid_json(output) else output
            _, failed = grade_rubric(reply, case["rubric"])
            reasons += ["rubric: " + f for f in failed]
        results.append({"id": case["id"], "category": case["category"], "critical": case.get("critical", False),
                        "passed": not reasons, "reasons": reasons})
    return results

def pass_rates(results):
    cats = {}
    for r in results:
        cats.setdefault(r["category"], []).append(r["passed"])
    return {c: sum(v) / len(v) for c, v in cats.items()}

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def regression_gate(base, new, max_drop=0.10):
    problems = []
    base_by_id = {r["id"]: r for r in base}
    flipped = [r["id"] for r in new if base_by_id[r["id"]]["passed"] and not r["passed"]]
    for r in new:
        if r["critical"] and not r["passed"]:
            problems.append(f"critical case {r['id']} fails: {r['reasons']}")
    old_rates, new_rates = pass_rates(base), pass_rates(new)
    for cat, old in old_rates.items():
        if new_rates.get(cat, 0) < old - max_drop:
            problems.append(f"{cat} dropped from {old:.0%} to {new_rates.get(cat, 0):.0%}")
    return len(problems) == 0, problems, flipped